# 01. Environment Setup & Validation

This notebook validates the development environment and ensures all components are properly configured for the LangChain RAG system.

## Objectives:
- Verify Python environment and dependencies
- Test Qdrant Cloud connection
- Validate Google Gemini API
- Check project structure
- Run initial health checks

## 1. Environment Setup

In [1]:
import sys
import os
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"Python version: {sys.version}")
print(f"Project root: {project_root}")
print(f"Source path: {src_path}")

Python version: 3.12.9 (main, Mar 17 2025, 21:36:21) [Clang 20.1.0 ]
Project root: /Users/sourangshupal/Documents/projects/langchain-rag-project
Source path: /Users/sourangshupal/Documents/projects/langchain-rag-project/src


## 2. Dependency Verification

In [2]:
!uv pip install google-genai

Using Python 3.12.9 environment at: /Users/sourangshupal/Documents/projects/langchain-rag-project/.venv
Audited 1 package in 26ms


In [3]:
# Add this debugging cell
import sys
import pkg_resources

# Check if google-genai is installed
try:
    dist = pkg_resources.get_distribution('google-genai')
    print(f"✅ google-genai version {dist.version} is installed")
    print(f"Location: {dist.location}")
except pkg_resources.DistributionNotFound:
    print("❌ google-genai not found")

# Check what's available in the google namespace
try:
    import google
    print(f"✅ google module found at: {google.__file__}")
    print(f"Available submodules: {dir(google)}")
except ImportError:
    print("❌ google module not found")

✅ google-genai version 1.31.0 is installed
Location: /Users/sourangshupal/Documents/projects/langchain-rag-project/.venv/lib/python3.12/site-packages
✅ google module found at: None
Available submodules: ['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


/var/folders/cj/13vbmk7n7fqgmdnqjjwn1bx80000gn/T/ipykernel_15582/1387358986.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Check core dependencies
dependencies = {
    'langchain': 'langchain',
    'qdrant_client': 'qdrant-client',
    'fastapi': 'fastapi',
    'uvicorn': 'uvicorn',
    'pydantic': 'pydantic',
    'pandas': 'pandas',
    'numpy': 'numpy'
}

missing_deps = []
for module, package in dependencies.items():
    try:
        __import__(module)
        print(f"✅ {package} - OK")
    except ImportError:
        print(f"❌ {package} - MISSING")
        missing_deps.append(package)

if missing_deps:
    print(f"\nMissing dependencies: {', '.join(missing_deps)}")
    print("Please install with: uv pip install -r requirements.txt")
else:
    print("\n🎉 All dependencies are installed!")

✅ langchain - OK
✅ qdrant-client - OK
✅ fastapi - OK
✅ uvicorn - OK
✅ pydantic - OK
✅ pandas - OK
✅ numpy - OK

🎉 All dependencies are installed!


In [5]:
# Check Gemini Packages 
try:
    import google.genai
    print("✅ google.genai import - SUCCESS")
    print(f"Available in google.genai: {dir(google.genai)}")
except ImportError as e:
    print(f"❌ google.genai import - FAILED: {e}")

# Also test the from import
try:
    from google import genai
    print("✅ from google import genai - SUCCESS")
except ImportError as e:
    print(f"❌ from google import genai - FAILED: {e}")

✅ google.genai import - SUCCESS
Available in google.genai: ['Client', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_adapters', '_api_client', '_api_module', '_base_url', '_common', '_extra_utils', '_live_converters', '_mcp_utils', '_replay_api_client', '_tokens_converters', '_transformers', 'batches', 'caches', 'chats', 'client', 'errors', 'files', 'live', 'live_music', 'models', 'operations', 'pagers', 'tokens', 'tunings', 'types', 'version']
✅ from google import genai - SUCCESS


## 3. Configuration Loading

In [6]:
import os
from dotenv import load_dotenv

# Load environment variables
env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print("✅ Environment variables loaded from .env")
else:
    print("⚠️  .env file not found. Using .env.example as template.")
    print("Please create .env file with your API keys.")

# Check required environment variables
required_vars = [
    'GOOGLE_API_KEY',
    'QDRANT_URL', 
    'QDRANT_API_KEY'
]

missing_vars = []
for var in required_vars:
    if os.getenv(var):
        print(f"✅ {var} - Set")
    else:
        print(f"❌ {var} - Missing")
        missing_vars.append(var)

if missing_vars:
    print(f"\n⚠️  Please set missing environment variables: {', '.join(missing_vars)}")

✅ Environment variables loaded from .env
✅ GOOGLE_API_KEY - Set
✅ QDRANT_URL - Set
✅ QDRANT_API_KEY - Set


## 4. Qdrant Connection Test

In [7]:
try:
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams
    
    # Initialize Qdrant client
    qdrant_url = os.getenv('QDRANT_URL')
    qdrant_api_key = os.getenv('QDRANT_API_KEY')
    
    if qdrant_url and qdrant_api_key:
        client = QdrantClient(
            url=qdrant_url,
            api_key=qdrant_api_key,
        )
        
        # Test connection
        collections = client.get_collections()
        print(f"✅ Qdrant connection successful!")
        print(f"Available collections: {len(collections.collections)}")
        
        # Check if our collection exists
        collection_name = os.getenv('COLLECTION_NAME', 'langchain_docs')
        collection_exists = any(c.name == collection_name for c in collections.collections)
        
        if collection_exists:
            print(f"✅ Collection '{collection_name}' already exists")
            info = client.get_collection(collection_name)
            print(f"   Vector count: {info.points_count}")
            print(f"   Vector size: {info.config.params.vectors.size}")
        else:
            print(f"ℹ️  Collection '{collection_name}' will be created during data ingestion")
    else:
        print("❌ Qdrant credentials not found. Please check your .env file.")
        
except Exception as e:
    print(f"❌ Qdrant connection failed: {str(e)}")

✅ Qdrant connection successful!
Available collections: 1
ℹ️  Collection 'langchain_docs' will be created during data ingestion


## 5. Google Gemini API Test

In [8]:
try:
    from google import genai  # New import style
    from google.genai import types  # For configuration types
    
    google_api_key = os.getenv('GOOGLE_API_KEY')
    
    if google_api_key:
        # Initialize client instead of using configure
        client = genai.Client(api_key=google_api_key)
        
        # Test embedding model
        try:
            embedding_model = "gemini-embedding-001"  # Model name without 'models/' prefix
            test_text = "Hello, this is a test for embedding generation."
            
            # Use client.models.embed_content instead of genai.embed_content
            result = client.models.embed_content(
                model=embedding_model,
                contents=test_text  # Note: 'contents' not 'content'
            )
            
            # Access embeddings from the response
            if result.embeddings and len(result.embeddings) > 0:
                embedding_dim = len(result.embeddings[0].values)
                print(f"✅ Gemini Embedding API working!")
                print(f"   Embedding dimension: {embedding_dim}")
            else:
                print(f"❌ No embeddings returned")
            
        except Exception as e:
            print(f"❌ Gemini Embedding API failed: {str(e)}")
        
        # Test generative model
        try:
            # Use client.models.generate_content instead of GenerativeModel
            response = client.models.generate_content(
                model='gemini-2.0-flash-001',  # Or 'gemini-2.0-flash' 
                contents="Say hello in one word."
            )
            print(f"✅ Gemini LLM API working!")
            print(f"   Test response: {response.text.strip()}")
            
        except Exception as e:
            print(f"❌ Gemini LLM API failed: {str(e)}")
            
    else:
        print("❌ Google API key not found. Please check your .env file.")
        
except ImportError as e:
    print(f"❌ Google GenAI package not installed: {str(e)}")
    print("   Install with: pip install google-genai")
except Exception as e:
    print(f"❌ Google Generative AI setup failed: {str(e)}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Gemini Embedding API working!
   Embedding dimension: 3072
✅ Gemini LLM API working!
   Test response: Hi.


## 6. Project Structure Validation

In [9]:
# Check project directory structure
expected_dirs = [
    'src', 'app', 'config', 'data', 'notebooks',
    'data/raw', 'data/processed', 'data/embeddings',
    'src/ingestion', 'src/chunking', 'src/embedding',
    'src/retrieval', 'src/generation', 'src/optimization'
]

expected_files = [
    'requirements.txt', '.env.example', '.gitignore',
    'config/config.yaml', 'src/config.py'
]

print("📁 Directory Structure:")
missing_dirs = []
for dir_path in expected_dirs:
    full_path = project_root / dir_path
    if full_path.exists():
        print(f"  ✅ {dir_path}/")
    else:
        print(f"  ❌ {dir_path}/")
        missing_dirs.append(dir_path)

print("\n📄 Required Files:")
missing_files = []
for file_path in expected_files:
    full_path = project_root / file_path
    if full_path.exists():
        print(f"  ✅ {file_path}")
    else:
        print(f"  ❌ {file_path}")
        missing_files.append(file_path)

if missing_dirs or missing_files:
    print(f"\n⚠️  Missing components detected. Please create them before proceeding.")
else:
    print(f"\n🎉 Project structure is complete!")

📁 Directory Structure:
  ✅ src/
  ✅ app/
  ✅ config/
  ✅ data/
  ✅ notebooks/
  ✅ data/raw/
  ✅ data/processed/
  ✅ data/embeddings/
  ✅ src/ingestion/
  ✅ src/chunking/
  ✅ src/embedding/
  ✅ src/retrieval/
  ✅ src/generation/
  ✅ src/optimization/

📄 Required Files:
  ✅ requirements.txt
  ✅ .env.example
  ✅ .gitignore
  ✅ config/config.yaml
  ✅ src/config.py

🎉 Project structure is complete!


## 7. System Health Summary

In [10]:
print("📋 SYSTEM HEALTH SUMMARY")
print("=" * 50)

health_checks = {
    "Python Environment": len(missing_deps) == 0 if 'missing_deps' in locals() else False,
    "Environment Variables": len(missing_vars) == 0 if 'missing_vars' in locals() else False,
    "Project Structure": (len(missing_dirs) == 0 and len(missing_files) == 0) if 'missing_dirs' in locals() and 'missing_files' in locals() else False,
    "Qdrant Connection": 'client' in locals(),
    "Google Gemini API": 'genai' in locals() and google_api_key is not None
}

all_healthy = True
for check, status in health_checks.items():
    status_icon = "✅" if status else "❌"
    print(f"{status_icon} {check}")
    if not status:
        all_healthy = False

print("\n" + "=" * 50)
if all_healthy:
    print("🎉 ALL SYSTEMS GREEN! Ready to proceed to Phase 2.")
else:
    print("⚠️  Some components need attention. Please resolve issues before continuing.")
    print("\nNext steps:")
    print("1. Install missing dependencies: pip install -r requirements.txt")
    print("2. Create .env file with your API keys")
    print("3. Ensure project structure is complete")
    print("4. Verify API connections")

📋 SYSTEM HEALTH SUMMARY
✅ Python Environment
✅ Environment Variables
✅ Project Structure
✅ Qdrant Connection
✅ Google Gemini API

🎉 ALL SYSTEMS GREEN! Ready to proceed to Phase 2.


## Next Steps

If all checks pass, you're ready to proceed to:
- **Phase 2**: Document ingestion and processing
- **Notebook 02**: `data_ingestion.ipynb`

If any checks fail, please resolve the issues and re-run this notebook.